# Comprobar referencias después de un cálculo

El cálculo independiente produce media 2.5 y varianza poblacional 1.25.
Inspeccionamos sus referencias con Recorda y un índice local explícito, retiramos
una entrada y modificamos bytes de una salida, y restauramos ambos archivos.

El estado de ejecución se conserva. Un hash coincidente comprueba bytes locales;
no certifica identidad científica, autoría, retención permanente o replay.
Una referencia sin digest, una que no se puede resolver, una omisión por política
y una operación interrumpida son situaciones distintas. Los índices permanecen
fuera del journal y las bibliotecas científicas conservan sus objetos.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("experiments").resolve()))
import recorda
from run_reference_checks import prepare, run

workspace = Path("reference-example")
print(sys.executable)

In [ ]:
trial = prepare(workspace)
files = trial["files"]
record = trial["records"]["full"]
assert trial["scientific_result"]["mean"] == 2.5
assert trial["scientific_result"]["variance"] == 1.25

In [ ]:
intact = recorda.check_references(record, resolver=files)
assert len(intact["references"]) == 4
assert all(row["status"] == "matched" for row in intact["references"])
[(row["group"], row["field"], row["status"]) for row in intact["references"]]

In [ ]:
source_path = workspace / "native/samples.json"
source_bytes = source_path.read_bytes()
source_path.unlink()
missing = recorda.check_references(record, resolver=files)
assert {row["status"] for row in missing["references"]} == {"missing", "matched"}
assert missing["session_status"] == "succeeded"
[(row["group"], row["field"], row["status"]) for row in missing["references"]]

## Vista técnica solicitada explícitamente

El informe JSON conserva cada observación. La vista agrupa los problemas y obtiene
mensajes de SMonitor sin emitir eventos ni cambiar la política de la aplicación.
Sus mensajes explican observaciones técnicas; el cálculo mantiene media 2.5 y
varianza 1.25. La pérdida actual de un archivo no cambia su ejecución registrada.


In [ ]:
missing_view = recorda.inspection_view(record, reference_report=missing)
assert missing_view["execution"]["status"] == "succeeded"
assert missing_view["reference_report"] == missing
(missing_finding,) = missing_view["findings"]
assert missing_finding["count"] == 2 and missing_finding["distinct_references"] == 1
assert missing_finding["presentation"]["status"] == "resolved"
{
    "execution": missing_view["execution"]["status"],
    "affected_occurrences": missing_finding["count"],
    "distinct_references": missing_finding["distinct_references"],
    "message": missing_finding["presentation"]["message"],
    "hint": missing_finding["presentation"]["hint"],
    "source_indices": missing_finding["source_indices"],
}

In [ ]:
source_path.write_bytes(source_bytes)
result_path = workspace / "native/result.json"
result_bytes = result_path.read_bytes()
result_path.write_bytes(result_bytes + b"\n")
modified = recorda.check_references(record, resolver=files)
assert {row["status"] for row in modified["references"]} == {"mismatched", "matched"}
[(row["group"], row["field"], row["status"]) for row in modified["references"]]

In [ ]:
modified_view = recorda.inspection_view(record, reference_report=modified)
assert modified_view["execution"]["status"] == "succeeded"
assert modified_view["reference_report"] == modified
(changed,) = modified_view["findings"]
assert changed["state"] == "mismatched" and changed["count"] == 2
assert changed["distinct_references"] == 1
changed["presentation"]

In [ ]:
result_path.write_bytes(result_bytes)
restored = recorda.check_references(record, resolver=files)
assert all(row["status"] == "matched" for row in restored["references"])
assert record.status == "succeeded"

In [ ]:
limits = recorda.check_references(trial["records"]["limits"], resolver=files)
minimal = recorda.check_references(trial["records"]["minimal"], resolver=files)
interrupted = recorda.check_references(trial["records"]["interrupted"], resolver=files)
assert {row["status"] for row in limits["references"]} == {"available_unverified", "unresolved"}
assert minimal["references"] == [] and len(minimal["omissions"]) == 6
assert interrupted["session_status"] == "incomplete" and interrupted["incomplete_operations"]
{
    "reference_limits": [row["status"] for row in limits["references"]],
    "omissions": minimal["omissions"],
    "incomplete": interrupted["incomplete_operations"],
}

In [ ]:
acceptance = run(workspace / "checked-simulation")
assert acceptance["scientific_result"] == trial["scientific_result"]
list(acceptance["reports"])

## Cobertura, comprobaciones ausentes y trabajo incompleto

Las seis omisiones de la política mínima son cobertura explícita. Un archivo sin
digest y una referencia sin vínculo local no equivalen a bytes alterados. Solicitar
la vista sin informe no comprueba archivos. El ejemplo de cola truncada que sigue
crea una copia dañada de un journal interrumpido; no atribuye una causa real.


In [ ]:
minimal_view = recorda.inspection_view(trial["records"]["minimal"], reference_report=minimal)
limits_view = recorda.inspection_view(trial["records"]["limits"], reference_report=limits)
interrupted_view = recorda.inspection_view(
    trial["records"]["interrupted"], reference_report=interrupted
)
unchecked_view = recorda.inspection_view(record)
assert minimal_view["coverage"]["omissions_by_reason"] == {"capture_policy": 6}
assert minimal_view["findings"] == []
assert unchecked_view["references"]["check_state"] == "not_requested"
assert unchecked_view["references"]["checked_occurrences"] is None
assert interrupted_view["execution"]["status"] == "incomplete"
tail_path = workspace / "truncated-copy.jsonl"
tail_path.write_bytes((workspace / "interrupted.jsonl").read_bytes() + b'{"unfinished')
tail = recorda.inspect(tail_path)
tail_view = recorda.inspection_view(tail, reference_report=recorda.check_references(tail))
assert tail_view["record"]["problems"] == ["truncated_tail"]
{
    "intentional_omissions": minimal_view["coverage"]["omissions_by_reason"],
    "limits": [f["presentation"] for f in limits_view["findings"]],
    "incomplete": interrupted_view["findings"],
    "truncated_prefix": tail_view["findings"],
    "without_checking": unchecked_view["references"],
}